# Multi-Model Benchmark

This notebook compares Logistic Regression, Random Forest, XGBoost, and a compact PyTorch MLP on the same validation split. PR-AUC is the primary model-selection metric. The held-out test set remains untouched.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_curve, roc_curve
from torch import nn

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_dataset, split_dataset
from src.evaluation import best_f1_threshold, classification_metrics
from src.models import FraudMLP, build_logistic_regression, build_random_forest, build_xgboost
from src.preprocessing import build_linear_preprocessor, build_tree_preprocessor, fit_transform_splits

random_state = 42
np.random.seed(random_state)
torch.manual_seed(random_state)

In [ ]:
transaction_dataset = load_dataset(project_root / "data" / "creditcard.csv")

(
    training_features,
    validation_features,
    test_features,
    training_labels,
    validation_labels,
    test_labels,
) = split_dataset(transaction_dataset, random_state=random_state)

print(f"Train: {len(training_features):,}")
print(f"Validation: {len(validation_features):,}")
print(f"Test reserved: {len(test_features):,}")

In [ ]:
linear_preprocessor = build_linear_preprocessor(training_features.columns.tolist())
(
    linear_training_features,
    linear_validation_features,
    _,
    fitted_linear_preprocessor,
) = fit_transform_splits(
    linear_preprocessor,
    training_features,
    validation_features,
    test_features,
)

tree_preprocessor = build_tree_preprocessor(training_features.columns.tolist())
(
    tree_training_features,
    tree_validation_features,
    _,
    fitted_tree_preprocessor,
) = fit_transform_splits(
    tree_preprocessor,
    training_features,
    validation_features,
    test_features,
)

In [ ]:
def evaluate_model(model_name, validation_scores):
    selected_threshold = best_f1_threshold(validation_labels, validation_scores)
    metrics = classification_metrics(
        validation_labels,
        validation_scores,
        threshold=selected_threshold,
    )
    return {"model": model_name, **metrics}


benchmark_results = []
validation_scores_by_model = {}

In [ ]:
logistic_model = build_logistic_regression(random_state=random_state)
logistic_model.fit(linear_training_features, training_labels)
logistic_scores = logistic_model.predict_proba(linear_validation_features)[:, 1]

validation_scores_by_model["Logistic Regression"] = logistic_scores
benchmark_results.append(evaluate_model("Logistic Regression", logistic_scores))

In [ ]:
random_forest_model = build_random_forest(random_state=random_state)
random_forest_model.fit(tree_training_features, training_labels)
random_forest_scores = random_forest_model.predict_proba(tree_validation_features)[:, 1]

validation_scores_by_model["Random Forest"] = random_forest_scores
benchmark_results.append(evaluate_model("Random Forest", random_forest_scores))

In [ ]:
xgboost_model = build_xgboost(random_state=random_state)

fraud_count = int(training_labels.sum())
legitimate_count = len(training_labels) - fraud_count
xgboost_model.set_params(scale_pos_weight=legitimate_count / fraud_count)

xgboost_model.fit(tree_training_features, training_labels)
xgboost_scores = xgboost_model.predict_proba(tree_validation_features)[:, 1]

validation_scores_by_model["XGBoost"] = xgboost_scores
benchmark_results.append(evaluate_model("XGBoost", xgboost_scores))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

neural_model = FraudMLP(linear_training_features.shape[1]).to(device)

training_tensor = torch.tensor(linear_training_features, dtype=torch.float32, device=device)
training_label_tensor = torch.tensor(training_labels.to_numpy(), dtype=torch.float32, device=device)
validation_tensor = torch.tensor(linear_validation_features, dtype=torch.float32, device=device)

fraud_count = float(training_labels.sum())
legitimate_count = float(len(training_labels) - training_labels.sum())
positive_weight = torch.tensor([legitimate_count / fraud_count], device=device)

loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(neural_model.parameters(), lr=1e-3, weight_decay=1e-5)

batch_size = 4096
best_validation_pr_auc = -np.inf
best_model_state = None
patience = 5
epochs_without_improvement = 0
maximum_epochs = 30

for epoch in range(maximum_epochs):
    neural_model.train()
    permutation = torch.randperm(training_tensor.size(0), device=device)

    for batch_start in range(0, training_tensor.size(0), batch_size):
        batch_indices = permutation[batch_start:batch_start + batch_size]
        batch_features = training_tensor[batch_indices]
        batch_labels = training_label_tensor[batch_indices]

        optimizer.zero_grad()
        logits = neural_model(batch_features)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()

    neural_model.eval()
    with torch.no_grad():
        validation_logits = neural_model(validation_tensor)
        neural_scores = torch.sigmoid(validation_logits).cpu().numpy()

    neural_metrics = classification_metrics(validation_labels, neural_scores, threshold=0.5)
    validation_pr_auc = neural_metrics["pr_auc"]

    if validation_pr_auc > best_validation_pr_auc + 1e-6:
        best_validation_pr_auc = validation_pr_auc
        best_model_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in neural_model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        break

neural_model.load_state_dict(best_model_state)
neural_model.eval()

with torch.no_grad():
    neural_scores = torch.sigmoid(neural_model(validation_tensor)).cpu().numpy()

validation_scores_by_model["PyTorch MLP"] = neural_scores
benchmark_results.append(evaluate_model("PyTorch MLP", neural_scores))

In [ ]:
benchmark_table = (
    pd.DataFrame(benchmark_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

benchmark_table[["model", "pr_auc", "roc_auc", "precision", "recall", "f1", "threshold", "fp", "fn", "tp"]]

In [ ]:
figure_directory = project_root / "outputs" / "figures"
metrics_directory = project_root / "outputs" / "metrics"
figure_directory.mkdir(parents=True, exist_ok=True)
metrics_directory.mkdir(parents=True, exist_ok=True)

plt.figure()
for model_name, model_scores in validation_scores_by_model.items():
    precision_values, recall_values, _ = precision_recall_curve(validation_labels, model_scores)
    plt.plot(recall_values, precision_values, label=model_name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision-recall curves")
plt.legend()
plt.tight_layout()
plt.savefig(figure_directory / "validation_precision_recall_curves.png", dpi=160)
plt.show()

In [ ]:
plt.figure()
for model_name, model_scores in validation_scores_by_model.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(validation_labels, model_scores)
    plt.plot(false_positive_rate, true_positive_rate, label=model_name)

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Validation ROC curves")
plt.legend()
plt.tight_layout()
plt.savefig(figure_directory / "validation_roc_curves.png", dpi=160)
plt.show()

In [ ]:
benchmark_table.to_csv(metrics_directory / "validation_model_benchmark.csv", index=False)

selected_model_row = benchmark_table.iloc[0]
selected_model = {
    "model": selected_model_row["model"],
    "selection_metric": "pr_auc",
    "selection_split": "validation",
    "pr_auc": float(selected_model_row["pr_auc"]),
    "roc_auc": float(selected_model_row["roc_auc"]),
    "threshold": float(selected_model_row["threshold"]),
}

(metrics_directory / "selected_validation_model.json").write_text(
    json.dumps(selected_model, indent=2),
    encoding="utf-8",
)

selected_model

## Interpretation

The model with the highest validation PR-AUC is selected as the leading candidate. Threshold-dependent metrics are reported at each model's validation-selected F1 threshold, but PR-AUC remains the primary discrimination metric. The held-out test set is still untouched.